# IOAI — 2026 Local Onia Churn (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-churn.zip', '/tmp/churn.zip')
    with zipfile.ZipFile('/tmp/churn.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Churn — 고객 이탈 예측 (RoAI/ONIA 2026 Local)

통신사 고객 데이터로 이탈 여부를 예측한다. 서브태스크 3개:
1. (결정적) 제공된 규칙으로 파생값 계산
2. (결정적) 제공된 규칙으로 파생값 계산
3. (ML) `Churn` 이탈 **확률** 예측 — ROC-AUC 로 채점

**제출**: `submission.csv` 롱포맷 `id, subtaskID, answer`.
**채점(로컬)**: 공식 test 라벨은 비공개라 train 의 결정적 held-out 으로 ST3 ROC-AUC 채점.
**데이터**: `data/train.csv`(+`data/test.csv`). 원본: platform.olimpiada-ai.ro/competitions/12

In [ ]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier

root_path = "data"
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df  = pd.read_csv(f"{root_path}/test.csv")
train_df.head()

In [ ]:
# 서브태스크 1: FinancialRiskScore = (Monthly Charge>70) + (Total Extra Data Charges>10)  (결정적)
# 서브태스크 2: ServiceQualityScore = (Avg Speed<50) + (Ping Score>80) + (Link Quality Index<30)  (결정적)
for d in (train_df, test_df):
    d["FinancialRiskScore"] = (d["Monthly Charge"]>70).astype(int) + (d["Total Extra Data Charges"]>10).astype(int)
    d["ServiceQualityScore"] = ((d["Avg Speed"]<50).astype(int) + (d["Ping Score"]>80).astype(int) + (d["Link Quality Index"]<30).astype(int))
subtask1 = test_df["FinancialRiskScore"]
subtask2 = test_df["ServiceQualityScore"]

In [ ]:
# 서브태스크 3: 이탈확률 예측 — 간단 베이스라인(수치형 컬럼 + 얕은 결정트리). 모범답안=RF, 모범답안2=표준화+로지스틱
# TODO: 범주형 더미인코딩·위경도 분해·결측 보정·튜닝으로 개선 (모범답안 Solution/ 참고)
feat = [c for c in train_df.select_dtypes(include=np.number).columns if c != "Churn"]
mean = train_df[feat].mean()
Xtr, ytr = train_df[feat].fillna(mean), train_df["Churn"]
Xte = test_df[feat].fillna(mean)
clf = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
subtask3 = clf.predict_proba(Xte)[:, 1]

In [ ]:
# 제출(롱포맷): id=SampleID, subtaskID, answer
test_ids = test_df["SampleID"]
def build_subtask(sid, ans):
    return pd.DataFrame({"id": test_ids, "subtaskID": sid, "answer": np.asarray(ans)})
submission = pd.concat([build_subtask(1, subtask1.astype(int)),
                        build_subtask(2, subtask2.astype(int)),
                        build_subtask(3, subtask3)])
submission.to_csv(f"{root_path}/submission.csv", index=False)
submission.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)